# Task 3 — Image Similarity Classification

This project addresses an image similarity task based on triplets of images.
A pretrained ResNet-18 is used to extract image embeddings, which are combined
into triplet representations and classified using a neural network.

The training data is augmented by swapping the candidate images to construct
positive and negative triplets. The resulting binary classifier predicts which
of two candidate images is more similar to a given reference image.

ETH Zürich — Introduction to Machine Learning, Spring 2024

In [ ]:
import numpy as np
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, TensorDataset
import os
import torch
from torchvision import transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
# Add any other imports you need here
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
!pip install tqdm
from tqdm import tqdm

In [2]:
# The device is automatically set to GPU if available, otherwise CPU
# If you want to force the device to CPU, you can change the line to
# device = torch.device("cpu")
# When using the GPU, it is important that your model and all data are on the 
# same device.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
def image_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

# Function to generate embeddings
def generate_embeddings():
    # Load the dataset and create DataLoader
    dataset = datasets.ImageFolder("dataset/", transform=image_transform())
    loader = DataLoader(dataset, batch_size=64, shuffle=False, pin_memory=True, num_workers=4)
    
    # Load pretrained model (ResNet-18)
    model = torchvision.models.resnet18(pretrained=True)
    model = nn.Sequential(*list(model.children())[:-1])  # Remove the last classification layer
    model.to(device)
    model.eval()

    # Initialize the embeddings array
    embedding_size = model[-1].in_features  # Check embedding size based on ResNet
    num_images = len(dataset)
    embeddings = np.zeros((num_images, embedding_size))
    
    # Extract embeddings
    with torch.no_grad():
        for idx, (data, _) in enumerate(loader):
            data = data.to(device)
            output = model(data).view(data.size(0), -1)  # Flatten the output
            start_idx = idx * 64
            end_idx = min(start_idx + 64, num_images)
            embeddings[start_idx:end_idx] = output.cpu().numpy()[:end_idx - start_idx]

    # Save embeddings
    np.save("dataset/embeddings.npy", embeddings)

In [4]:
# Function to get triplet data and generate features/labels
def get_data(triplets_file, train=True):
    # Load the triplets
    with open(triplets_file) as f:
        triplets = [line.strip().split() for line in f]

    # Load the dataset and embeddings
    dataset = datasets.ImageFolder("dataset/", transform=None)
    filenames = [os.path.splitext(os.path.basename(s[0]))[0] for s in dataset.samples]
    embeddings = np.load("dataset/embeddings.npy")
    
    # Create a mapping from filenames to embeddings
    file_to_embedding = {filename: embedding for filename, embedding in zip(filenames, embeddings)}

    # Create features and labels from triplets
    X = []
    y = []

    for triplet in triplets:
        emb = [file_to_embedding[name] for name in triplet]
        X.append(np.hstack([emb[0], emb[1], emb[2]]))
        y.append(1)  # Label for positive triplet

        # Augment the data with negative triplets
        if train:
            X.append(np.hstack([emb[0], emb[2], emb[1]]))
            y.append(0)  # Label for negative triplet

    X = np.vstack(X)
    y = np.hstack(y)

    return X, y


In [5]:
# Create DataLoader from numpy arrays
def create_loader_from_np(X, y=None, train=True, batch_size=64, shuffle=True, num_workers=4):
    if train:
        dataset = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).long())
    else:
        dataset = TensorDataset(torch.from_numpy(X).float())

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=True)


In [6]:
# Define a simple classifier
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3000, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))  # Sigmoid for binary classification

In [7]:
# Function to train the model
def train_model(train_loader):
    model = Net()
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()  # Binary CrossEntropy Loss

    # Training loop
    for epoch in range(10):  # 10 epochs
        model.train()  # Set model to training mode
        total_loss = 0
        for X, y in tqdm(train_loader, desc=f"Epoch {epoch + 1}/10"):
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()  # Zero the parameter gradients
            outputs = model(X).squeeze()  # Forward pass
            loss = criterion(outputs, y.float())  # Calculate loss
            loss.backward()  # Backward pass
            optimizer.step()  # Update weights
            total_loss += loss.item()  # Accumulate loss

        print(f"Epoch {epoch + 1} - Loss: {total_loss / len(train_loader)}")

    return model  # Return trained model

In [9]:
# Function to test the model
def test_model(model, test_loader):
    model.eval()  # Set model to evaluation mode
    predictions = []

    with torch.no_grad():
        for X in test_loader:
            X = X.to(device)
            outputs = model(X).cpu().numpy().squeeze()  # Convert to numpy and squeeze
            predictions.append((outputs > 0).astype(int))  # Convert to binary prediction

    return np.vstack(predictions)  # Return predictions as a 2D numpy array

# Main execution flow
if __name__ == "__main__":
    # Generate embeddings from the dataset
    if not os.path.exists("dataset/embeddings.npy"):
        generate_embeddings()

    # Load the training triplets
    TRAIN_TRIPLETS = "train_triplets.txt"
    X, y = get_data(TRAIN_TRIPLETS)
    train_loader = create_loader_from_np(X, y, train=True, batch_size=64)

    # Train the model
    model = train_model(train_loader)

    # Load the test triplets
    TEST_TRIPLETS = "test_triplets.txt"
    X_test = get_data(TEST_TRIPLETS, train=False)
    test_loader = create_loader_from_np(X_test, train=False, batch_size=2048)

    # Test the model and save predictions
    predictions = test_model(model, test_loader)
    np.savetxt("results.txt", predictions, fmt="%d")
    print("Results saved to results.txt")